# 0. feladat: Adatok előfeldolgozása

## Importok

In [28]:
import json
from sklearn.naive_bayes import GaussianNB
from simple_rules import AttackPlausibilityChecker
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from IDS import IntrusionDetector

## Adatbetöltés

In [29]:
with open('nslkdd_features.json', 'r') as f:
    features_meta = json.load(f)

column_names = [feat['name'] for feat in features_meta]

train_df = pd.read_csv('NSL-KDD/KDDTrain+.txt', header=None, names=column_names)
test_df = pd.read_csv('NSL-KDD/KDDTest+.txt', header=None, names=column_names)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(train_df['label'].value_counts())

Train shape: (125973, 43)
Test shape:  (22544, 43)
label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64


## Bináris címkék kezelése

In [30]:
#Eredeti címkék
train_original_labels = train_df['label'].copy()
test_original_labels = test_df['label'].copy()

#Bináris címkék: normál=0, támadás=1
train_df['binary_label'] = (train_df['label'] != 'normal').astype(int)
test_df['binary_label'] = (test_df['label'] != 'normal').astype(int)

print(f"\nTrain label distribution:\n{train_df['binary_label'].value_counts()}")
print(f"\nTest label distribution:\n{test_df['binary_label'].value_counts()}")


Train label distribution:
binary_label
0    67343
1    58630
Name: count, dtype: int64

Test label distribution:
binary_label
1    12833
0     9711
Name: count, dtype: int64


## *Label* és *difficulty* oszlopok eltávolítása

In [31]:
#Droppoljuk a labelt, mivel ez a célváltozó
#Droppoljuk a difficulty-t is - ez csak egy metaadat, ami azt mutatja meg milyen nehéz detektálni a mintát, tehát nem igazi feature
train_df = train_df.drop(columns=['label', 'difficulty'])
test_df = test_df.drop(columns=['label', 'difficulty'])

## One-hot kódolás a *protocol_type*, *service* és *flag* oszlopokra

In [32]:
#One-hot kódolás a következő 3 oszlopra
categorical_cols = ['protocol_type', 'service', 'flag']

#Tanító és teszt adathalmazok kombinálása a konzisztens one-hot kódoláshoz
combined = pd.concat([train_df[categorical_cols], test_df[categorical_cols]], axis=0)
combined_onehot = pd.get_dummies(combined, columns=categorical_cols)

#A kombinált adat szétválasztés tanító és teszt adathalmazra
train_onehot = combined_onehot.iloc[:len(train_df)].reset_index(drop=True)
test_onehot = combined_onehot.iloc[len(train_df):].reset_index(drop=True)

#Kategorikus oszlopok helyettesítése one-hot kódolt oszlopokkal
train_numeric = train_df.drop(columns=categorical_cols).reset_index(drop=True)
test_numeric = test_df.drop(columns=categorical_cols).reset_index(drop=True)

#Binary_label szétszedése
y_train = train_numeric['binary_label'].values
y_test = test_numeric['binary_label'].values
train_numeric = train_numeric.drop(columns=['binary_label'])
test_numeric = test_numeric.drop(columns=['binary_label'])

#Numerikus és one-hot enkódolt jellemzők
X_train_df = pd.concat([train_numeric, train_onehot], axis=1)
X_test_df = pd.concat([test_numeric, test_onehot], axis=1)

## Standardizáció

In [33]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df.values.astype(np.float32))
X_test = scaler.transform(X_test_df.values.astype(np.float32))

print(f"\nAfter standardization:")
print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"X_test shape:  {X_test.shape}, dtype: {X_test.dtype}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")
print(f"X_train mean: {X_train.mean():.6f}")
print(f"X_train std: {X_train.std():.6f}")


After standardization:
X_train shape: (125973, 122), dtype: float32
X_test shape:  (22544, 122), dtype: float32
y_train shape: (125973,), y_test shape: (22544,)
X_train mean: -0.000000
X_train std: 0.995893


# 1. feladat: Az IDS bináris osztályozó tanítása és értékelése

## Adatok előkészítése

In [34]:
#Konvertálás PyTorch tenzorokba
#Unsqueeze(1) azért szükséges, mert a címke alakjának (N,1)-nek kell lennie
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

## Modell, loss, optimalizáló inicializálása

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = IntrusionDetector(input_dim=X_train.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Tanítás

In [36]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    epoch_loss = running_loss / len(train_dataset)

    model.eval()
    with torch.no_grad():
        tp = (model(X_test_t.to(device)).cpu().numpy() >= 0.5).astype(int).flatten()
        ta = accuracy_score(y_test, tp)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Test acc: {ta:.4f}")
    model.train()

Epoch [1/10], Loss: 0.0805, Test acc: 0.7775
Epoch [2/10], Loss: 0.0320, Test acc: 0.7824
Epoch [3/10], Loss: 0.0248, Test acc: 0.7985
Epoch [4/10], Loss: 0.0226, Test acc: 0.7889
Epoch [5/10], Loss: 0.0221, Test acc: 0.7913
Epoch [6/10], Loss: 0.0210, Test acc: 0.7913
Epoch [7/10], Loss: 0.0206, Test acc: 0.7944
Epoch [8/10], Loss: 0.0184, Test acc: 0.7942
Epoch [9/10], Loss: 0.0198, Test acc: 0.7996
Epoch [10/10], Loss: 0.0186, Test acc: 0.7904


## Kiértékelés

In [37]:
model.eval()
with torch.no_grad():
    X_test_dev = X_test_t.to(device)
    y_pred_prob = model(X_test_dev).cpu().numpy()
    y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

y_true = y_test

## Teljes teszt halmaz eredmények

In [38]:
def print_metrics(name, y_true, y_pred):
    print(f"\n{name}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score:  {f1_score(y_true, y_pred):.4f}")

print_metrics("Full Test Set", y_true, y_pred)


Full Test Set
Accuracy:  0.7904
Precision: 0.9259
Recall:    0.6867
F1-score:  0.7886


## Zero-day vs nem zero-day támadások azonosítása

In [39]:
#Zero-day támadások: a trainben nem, de testben benne lévő támadás típusok
train_attack_types = set(train_original_labels[train_original_labels != 'normal'])
test_attack_types = set(test_original_labels[test_original_labels != 'normal'])
zero_day_types = test_attack_types - train_attack_types

print(f"\nTrain attack types: {sorted(train_attack_types)}")
print(f"Test attack types:  {sorted(test_attack_types)}")
print(f"Zero-day types:     {sorted(zero_day_types)}")

#Zero-day maszk: olyan test minták, amelyek eredeti címkéje egy zero-day támadás
is_zero_day = np.isin(test_original_labels, list(zero_day_types))
#Nem zero-day támadások: támadások, melyek szerepeltek a tanításban
is_non_zero_day_attack = (y_test == 1) & (~is_zero_day)


Train attack types: ['back', 'buffer_overflow', 'ftp_write', 'guess_passwd', 'imap', 'ipsweep', 'land', 'loadmodule', 'multihop', 'neptune', 'nmap', 'perl', 'phf', 'pod', 'portsweep', 'rootkit', 'satan', 'smurf', 'spy', 'teardrop', 'warezclient', 'warezmaster']
Test attack types:  ['apache2', 'back', 'buffer_overflow', 'ftp_write', 'guess_passwd', 'httptunnel', 'imap', 'ipsweep', 'land', 'loadmodule', 'mailbomb', 'mscan', 'multihop', 'named', 'neptune', 'nmap', 'perl', 'phf', 'pod', 'portsweep', 'processtable', 'ps', 'rootkit', 'saint', 'satan', 'sendmail', 'smurf', 'snmpgetattack', 'snmpguess', 'sqlattack', 'teardrop', 'udpstorm', 'warezmaster', 'worm', 'xlock', 'xsnoop', 'xterm']
Zero-day types:     ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']


## Zero-day támadások kiértékelése

In [40]:
zero_day_acc = accuracy_score(y_true[is_zero_day], y_pred[is_zero_day])
print(f"Zero-day accuracy: {zero_day_acc:.4f} ({is_zero_day.sum()} samples)")

Zero-day accuracy: 0.4784 (3750 samples)


## Nem zero-day támadások kiértékelése

In [41]:
non_zero_day_acc = accuracy_score(y_true[is_non_zero_day_attack], y_pred[is_non_zero_day_attack])
print(f"Non-zero-day accuracy: {non_zero_day_acc:.4f} ({is_non_zero_day_attack.sum()} samples)")

Non-zero-day accuracy: 0.7728 (9083 samples)


# 2. feladat: Evasion korlátozott PGD-vel

## Módosíthatósági maszk

In [42]:
#Eredeti jellezők párosítása a módosíthatóságukhoz
modifiability_map = {}
for feat in features_meta:
    modifiability_map[feat['name']] = feat['modifiability']

feature_names = list(X_train_df.columns)

#Mask a highly modifiable jelemzőknek
highly_modifiable_mask = np.zeros(len(feature_names), dtype=bool)
partially_modifiable_mask = np.zeros(len(feature_names), dtype=bool)

for i, fname in enumerate(feature_names):
    #One-hot kódolt esetben, a base név a "_" karakter előtt van
    for orig_name in modifiability_map:
        if fname == orig_name or fname.startswith(orig_name + '_'):
            mod_level = modifiability_map[orig_name]
            if mod_level == 'HIGHLY_MODIFIABLE':
                highly_modifiable_mask[i] = True
            elif mod_level == 'PARTIALLY_MODIFIABLE':
                partially_modifiable_mask[i] = True
            break

#Kombinált mask
combined_modifiable_mask = highly_modifiable_mask | partially_modifiable_mask

print(f"Total features: {len(feature_names)}")
print(f"Highly modifiable: {highly_modifiable_mask.sum()}")
print(f"Partially modifiable: {partially_modifiable_mask.sum()}")
print(f"Combined modifiable: {combined_modifiable_mask.sum()}")

Total features: 122
Highly modifiable: 4
Partially modifiable: 96
Combined modifiable: 100


## Támadás setup

In [43]:
#Feature tartományok meghatározása
feature_min = X_train.min(axis=0)
feature_max = X_train.max(axis=0)

#Helyesen besorolt támadási minták
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
with torch.no_grad():
    preds = model(X_test_t).cpu().numpy().flatten()
    pred_labels = (preds >= 0.5).astype(int)

#True pozitív támadások
true_positive_mask = (y_test == 1) & (pred_labels == 1)
tp_indices = np.where(true_positive_mask)[0]
print(f"\nCorrectly classified attacks (true positives): {len(tp_indices)}")

#Támadástípusok meghatározása
attack_type_map = {}
for feat in features_meta:
    if feat['name'] == 'label':
        for label, category in feat['flag_values'].items():
            if 'DoS' in category:
                attack_type_map[label] = 'DoS'
            elif 'Probe' in category:
                attack_type_map[label] = 'Probe'
            elif 'R2L' in category:
                attack_type_map[label] = 'R2L'
            elif 'U2R' in category:
                attack_type_map[label] = 'U2R'



Correctly classified attacks (true positives): 8813


## PGD támadás

In [44]:
def pgd_attack(model, x_orig, modifiable_mask, epsilon, alpha=0.01,
               num_iter=40, feat_min=None, feat_max=None):

    x_adv = torch.tensor(x_orig.copy(), dtype=torch.float32, device=device).unsqueeze(0)
    x_orig_t = torch.tensor(x_orig.copy(), dtype=torch.float32, device=device).unsqueeze(0)

    mask_t = torch.tensor(modifiable_mask, dtype=torch.float32, device=device)
    feat_min_t = torch.tensor(feat_min, dtype=torch.float32, device=device)
    feat_max_t = torch.tensor(feat_max, dtype=torch.float32, device=device)

    for _ in range(num_iter):
        x_adv.requires_grad_(True)
        output = model(x_adv)

        #A kimenetet szeretnénk minimalizálni
        loss = output
        loss.backward()

        with torch.no_grad():
            grad = x_adv.grad.sign()
            #Maszkolás
            perturbation = -alpha * grad * mask_t
            x_adv = x_adv + perturbation

            delta = x_adv - x_orig_t
            delta = torch.clamp(delta, -epsilon, epsilon)
            #Újramaszkolás
            delta = delta * mask_t
            x_adv = x_orig_t + delta

            #Valid tartományokba való szorítás
            x_adv = torch.max(x_adv, feat_min_t.unsqueeze(0))
            x_adv = torch.min(x_adv, feat_max_t.unsqueeze(0))

            x_adv = x_adv.detach()

    return x_adv.squeeze(0).cpu().numpy()

## Támadás

In [45]:
epsilons = [0.05, 0.1, 0.15, 0.2, 0.3]
checker = AttackPlausibilityChecker(feature_names)

print(f"PGD Evasion Attack Results")

for eps in epsilons:
    successful_attacks = 0
    plausible_attacks = 0
    total_attacks = len(tp_indices)

    for idx in tp_indices:
        x_orig = X_test[idx]
        orig_label = test_original_labels[idx]

        atk_type = attack_type_map.get(orig_label, 'DoS')

        #Először highly_modifiable jellemzőkkel futtatás
        x_adv = pgd_attack(model, x_orig, highly_modifiable_mask,
                           epsilon=eps, alpha=0.01, num_iter=40,
                           feat_min=feature_min, feat_max=feature_max)

        x_adv_t = torch.tensor(x_adv, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            pred = model(x_adv_t).item()

        #Ha nem sikeres így, próbáljuk meg a kombinálttal
        if pred >= 0.5:
            x_adv = pgd_attack(model, x_orig, combined_modifiable_mask,
                               epsilon=eps, alpha=0.01, num_iter=40,
                               feat_min=feature_min, feat_max=feature_max)
            x_adv_t = torch.tensor(x_adv, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                pred = model(x_adv_t).item()

        if pred < 0.5:
            successful_attacks += 1

            #Plauzabilitás ellenőrzése
            is_plausible, violations = checker.check_plausibility(
                x_adv, attack_type=atk_type, scaler=scaler, verbose=False
            )
            if is_plausible:
                plausible_attacks += 1

    success_rate = successful_attacks / total_attacks * 100 if total_attacks > 0 else 0
    plausible_rate = plausible_attacks / successful_attacks * 100 if successful_attacks > 0 else 0

    print(f"\nEpsilon: {eps}")
    print(f"  Total attacks attempted:  {total_attacks}")
    print(f"  Successful attacks:       {successful_attacks} ({success_rate:.2f}%)")
    print(f"  Plausible attacks:        {plausible_attacks} ({plausible_rate:.2f}% of successful)")

PGD Evasion Attack Results

Epsilon: 0.05
  Total attacks attempted:  8813
  Successful attacks:       280 (3.18%)
  Plausible attacks:        31 (11.07% of successful)

Epsilon: 0.1
  Total attacks attempted:  8813
  Successful attacks:       589 (6.68%)
  Plausible attacks:        126 (21.39% of successful)

Epsilon: 0.15
  Total attacks attempted:  8813
  Successful attacks:       712 (8.08%)
  Plausible attacks:        139 (19.52% of successful)

Epsilon: 0.2
  Total attacks attempted:  8813
  Successful attacks:       905 (10.27%)
  Plausible attacks:        208 (22.98% of successful)

Epsilon: 0.3
  Total attacks attempted:  8813
  Successful attacks:       1189 (13.49%)
  Plausible attacks:        252 (21.19% of successful)


# 3. feladat: PGD plauzibilitási loss-szal

## 1. lépés: GNB modell tanítása

In [46]:
#Eredeti multi-class címkékkel való tanítás
gnb = GaussianNB()
gnb.fit(X_train, train_original_labels)

#GNB paraméterek kinyerése -> Ez a differenciálható log-likelihood számításhoz kell
#gnb.theta_ -> átlag
#gnb.var_ -> variancia
gnb_classes = list(gnb.classes_)
gnb_means = torch.tensor(gnb.theta_, dtype=torch.float32, device=device)
gnb_vars = torch.tensor(gnb.var_, dtype=torch.float32, device=device)

print(f"GNB trained with {len(gnb_classes)} classes: {gnb_classes}")

tp_indices_nozd = tp_indices[~np.isin(test_original_labels[tp_indices], list(zero_day_types))]
print(f"True positives (non-zero-day): {len(tp_indices_nozd)}")

GNB trained with 23 classes: [np.str_('back'), np.str_('buffer_overflow'), np.str_('ftp_write'), np.str_('guess_passwd'), np.str_('imap'), np.str_('ipsweep'), np.str_('land'), np.str_('loadmodule'), np.str_('multihop'), np.str_('neptune'), np.str_('nmap'), np.str_('normal'), np.str_('perl'), np.str_('phf'), np.str_('pod'), np.str_('portsweep'), np.str_('rootkit'), np.str_('satan'), np.str_('smurf'), np.str_('spy'), np.str_('teardrop'), np.str_('warezclient'), np.str_('warezmaster')]
True positives (non-zero-day): 7019


## 2. és 3. lépés: Kombinált loss és plauzibilitási loss

In [47]:
def gnb_log_likelihood(x, class_label):
    class_idx = gnb_classes.index(class_label)
    mu = gnb_means[class_idx]
    var = gnb_vars[class_idx]

    log_probs = -0.5 * torch.log(2 * np.pi * var) - (x.squeeze(0) - mu) ** 2 / (2 * var)
    return log_probs.sum()

def pgd_attack_with_plausibility(model, x_orig, modifiable_mask, epsilon,
                                 attack_label, alpha=0.01, num_iter=40,
                                 feat_min=None, feat_max=None, lam=0.1):
    x_adv = torch.tensor(x_orig.copy(), dtype=torch.float32, device=device).unsqueeze(0)
    x_orig_t = torch.tensor(x_orig.copy(), dtype=torch.float32, device=device).unsqueeze(0)

    mask_t = torch.tensor(modifiable_mask, dtype=torch.float32, device=device)
    feat_min_t = torch.tensor(feat_min, dtype=torch.float32, device=device)
    feat_max_t = torch.tensor(feat_max, dtype=torch.float32, device=device)

    for _ in range(num_iter):
        x_adv.requires_grad_(True)

        output = model(x_adv)

        target = torch.zeros_like(output)
        L_classifier = nn.functional.binary_cross_entropy(output, target)

        log_lik = gnb_log_likelihood(x_adv, attack_label)
        L_plausibility = -log_lik

        #Kombinált loss
        L_total = L_classifier + lam * L_plausibility
        L_total.backward()

        with torch.no_grad():
            grad = x_adv.grad.sign()
            perturbation = -alpha * grad * mask_t
            x_adv = x_adv + perturbation

            delta = x_adv - x_orig_t
            delta = torch.clamp(delta, -epsilon, epsilon)
            delta = delta * mask_t
            x_adv = x_orig_t + delta

            #Valid tartományokba való szorítás
            x_adv = torch.max(x_adv, feat_min_t.unsqueeze(0))
            x_adv = torch.min(x_adv, feat_max_t.unsqueeze(0))

            x_adv = x_adv.detach()

    return x_adv.squeeze(0).cpu().numpy()

## 4. lépés: Implementáció és kiértékelés

In [48]:
epsilons = [0.05, 0.1, 0.15, 0.2, 0.3]
checker = AttackPlausibilityChecker(feature_names)

print(f"PGD with Plausibility Loss Results")

for eps in epsilons:
    successful_attacks = 0
    plausible_attacks = 0
    total_attacks = len(tp_indices_nozd)

    for idx in tp_indices_nozd:
        x_orig = X_test[idx]
        orig_label = test_original_labels[idx]
        atk_type = attack_type_map.get(orig_label, 'DoS')

        #Először highly_modifiable
        x_adv = pgd_attack_with_plausibility(
            model, x_orig, highly_modifiable_mask, epsilon=eps,
            attack_label=orig_label, alpha=0.01, num_iter=40,
            feat_min=feature_min, feat_max=feature_max, lam=0.1
        )

        x_adv_t = torch.tensor(x_adv, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            pred = model(x_adv_t).item()

        #Utána kombinált mask
        if pred >= 0.5:
            x_adv = pgd_attack_with_plausibility(
                model, x_orig, combined_modifiable_mask, epsilon=eps,
                attack_label=orig_label, alpha=0.01, num_iter=40,
                feat_min=feature_min, feat_max=feature_max, lam=0.1
            )
            x_adv_t = torch.tensor(x_adv, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                pred = model(x_adv_t).item()

        if pred < 0.5:
            successful_attacks += 1

            is_plausible, violations = checker.check_plausibility(
                x_adv, attack_type=atk_type, scaler=scaler, verbose=False
            )
            if is_plausible:
                plausible_attacks += 1

    success_rate = successful_attacks / total_attacks * 100 if total_attacks > 0 else 0
    plausible_rate = plausible_attacks / successful_attacks * 100 if successful_attacks > 0 else 0

    print(f"\nEpsilon: {eps}")
    print(f"  Total attacks attempted:  {total_attacks}")
    print(f"  Successful attacks:       {successful_attacks} ({success_rate:.2f}%)")
    print(f"  Plausible attacks:        {plausible_attacks} ({plausible_rate:.2f}% of successful)")

PGD with Plausibility Loss Results

Epsilon: 0.05
  Total attacks attempted:  7019
  Successful attacks:       24 (0.34%)
  Plausible attacks:        7 (29.17% of successful)

Epsilon: 0.1
  Total attacks attempted:  7019
  Successful attacks:       66 (0.94%)
  Plausible attacks:        26 (39.39% of successful)

Epsilon: 0.15
  Total attacks attempted:  7019
  Successful attacks:       124 (1.77%)
  Plausible attacks:        35 (28.23% of successful)

Epsilon: 0.2
  Total attacks attempted:  7019
  Successful attacks:       193 (2.75%)
  Plausible attacks:        42 (21.76% of successful)

Epsilon: 0.3
  Total attacks attempted:  7019
  Successful attacks:       346 (4.93%)
  Plausible attacks:        70 (20.23% of successful)


# 4. feladat: Elemzési kérdések

## 1. feladat: 2. és 3. feladat eredményeinek összehasonlítása

A 2. feladatban az epsilon növelésével egyre több támadás sikeres (3%→13%), a plauzibilitási
arány pedig viszonylag stabilan 11-23% között mozog. A 3. feladatban lényegesen kevesebb
támadás sikeres (0.3%→5%), mivel a plausibility loss visszahúzza a mintát a valós támadási
eloszlás felé, ami nehezíti a döntési határ átlépését.

A plauzibilitási arányokat összehasonlítva azt láthatjuk, hogy kis epsilonoknál (0.05, 0.1) a Task 3
egyértelműen jobb — 29-39% vs 11-21%. Ez azért van, mert a GNB loss közvetlenül részt
vesz az optimalizációban, és a gradiens egyszerre próbálja csökkenteni a classifier
outputot és közel tartani a mintát az eredeti támadástípus eloszlásához. Nagyobb
epsilonoknál (0.2, 0.3) a két módszer plauzibilitási aránya közelít egymáshoz
(kb. 20-21%), de a 3. feladat sikeres támadásai összességében kisebb perturbációval érik
el a kijátszást, tehát minőségileg jobbak.

# 2. feladat: Miért jobb a PGD az alternatíváknál?


**A alternatíva (utólagos szűrés):** Itt lefuttatjuk a sima PGD-t, majd utólag eldobjuk
azokat a támadásokat, amelyek nem mennek át a plauzibilitási ellenőrzésen. A probléma az,
hogy a gradiens semmit nem tud arról, hogy plauzibilisnek kellene lennie az eredménynek.
Lényegében random, hogy egy sikeres támadás átmegy-e a szabályokon vagy sem. Nagy
epszilonoknál szinte mindent eldobunk, kicsi epszilonoknál meg alig van sikeres támadás,
így mindkét esetben nagyon kevés használható példát kapunk.

**B alternatíva (lépésenkénti elutasítás):** Minden PGD iteráció után ellenőrizzük a
plauzibilitást, és ha sérti, visszautasítjuk a lépést. Ezzel az a baj, hogy a plauzibilitási
szabályok lényegében igen/nem döntések, amiből a gradiens nem tud tanulni. Ha a gradiens
irány nem plauzibilis régióba visz, az optimalizáció beragad, mert ugyanabba az irányba
próbálna lépni újra és újra.

A 3. feladat megközelítése mindkét problémát megoldja, mert a plausibility loss
differenciálható. A gradiens egyszerre kap információt mindkét célról, és a λ
paraméterrel szabályozható az egyensúly közöttük. Nem kell utólag szűrni, nem ragad
be az optimalizáció, hanem folyamatosan egy kompromisszumot keres a döntési határ
átlépése és a valószerűség között.

# Eredménytáblázatok

### IDS teljesítmény (1. feladat)

| Részhalmaz | Accuracy | Precision | Recall | F1-score |
|---|---|---|---|---|
| Teljes teszt halmaz | 0.7904 | 0.9259 | 0.6867 | 0.7886 |

| Részhalmaz | Accuracy |
|---|---|
| Zero-day támadások (3750 minta) | 0.4784 |
| Non-zero-day támadások (9083 minta) | 0.7728 |

### 2. feladat: Korlátozott PGD (8813 true positive mintán)

| Epsilon | Sikeres támadások | Siker arány | Plauzibilis támadások | Plauzibilis arány |
|---|---|---|---|---|
| 0.05 | 280 | 3.18% | 31 | 11.07% |
| 0.10 | 589 | 6.68% | 126 | 21.39% |
| 0.15 | 712 | 8.08% | 139 | 19.52% |
| 0.20 | 905 | 10.27% | 208 | 22.98% |
| 0.30 | 1189 | 13.49% | 252 | 21.19% |

### 3. feladat: PGD plausibility loss-szal (7019 non-zero-day true positive mintán)

| Epsilon | Sikeres támadások | Siker arány | Plauzibilis támadások | Plauzibilis arány |
|---|---|---|---|---|
| 0.05 | 24 | 0.34% | 7 | 29.17% |
| 0.10 | 66 | 0.94% | 26 | 39.39% |
| 0.15 | 124 | 1.77% | 35 | 28.23% |
| 0.20 | 193 | 2.75% | 42 | 21.76% |
| 0.30 | 346 | 4.93% | 70 | 20.23% |